# 04 — Comparer les PNG et mesurer les positions

**Objectif :** examiner les générations appariées sans confondre MSE, qualité visuelle et mesure géométrique.

Le protocole courant génère 24 scènes × 3 graines × 4 méthodes = **288 PNG**, dont **144 images du candidat**. Le nombre effectivement présent est lu dans le manifeste. Le 8B et le candidat partagent le diffuseur, la graine, le nombre de pas et la résolution.

| Méthode | Conditionnement |
|---|---|
| teacher_text | 8B, prompt naturel |
| candidate_text | 4B FP8 + projecteur, prompt naturel |
| teacher_layout | 8B, prompt avec coordonnées |
| candidate_layout | 4B FP8 + projecteur et boîtes |

Les modes naturels et les modes avec coordonnées sont analysés séparément.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


In [ ]:
from collections import Counter

predictions_path = result_file("run/validation/predictions.json")
predictions = read_json(predictions_path, {"rows": [], "config": {}})
counts = Counter(r["method"] for r in predictions["rows"])
table([{"méthode": method, "images": count} for method, count in sorted(counts.items())],
      ["méthode", "images"])
if not counts:
    print("La galerie n'est pas encore disponible dans cette copie.")

## Galerie paginée sur damier

Changer PAGE pour parcourir **tous** les groupes. Chaque groupe contient une scène, une graine et les quatre méthodes. Le damier est composé uniquement à l'affichage ; les pixels du PNG source restent inchangés.

Pour chaque candidat : vérifier le nombre d'objets, leur identité, leur séparation, la relation demandée, la position et la transparence. Une première page ne vaut pas revue complète.

In [ ]:
PAGE = 0
GROUPS_PER_PAGE = 4
groups = sorted({(r["scene_id"], r["seed"]) for r in predictions["rows"]})
selected = groups[PAGE * GROUPS_PER_PAGE:(PAGE + 1) * GROUPS_PER_PAGE]
print("Page", PAGE + 1, "/", max(1, (len(groups) + GROUPS_PER_PAGE - 1) // GROUPS_PER_PAGE))
for scene, seed in selected:
    members = [r for r in predictions["rows"] if (r["scene_id"], r["seed"]) == (scene, seed)]
    print("\nScène :", scene, "| graine :", seed)
    if members:
        print("Prompt :", members[0]["prompt"])
    for method in ("teacher_text", "candidate_text", "teacher_layout", "candidate_layout"):
        row = next((r for r in members if r["method"] == method), None)
        if row:
            path = contained(predictions_path.parent, row["file_name"])
            if sha256(path) != row["image_sha256"]:
                raise ValueError("PNG modifié depuis le manifeste.")
            show_image(path, method)

## Revue visuelle et évaluation statistique

La revue de publication identifie son auteur et sa méthode : `agent_visual_review` ou `human_visual_review`. Elle énumère toutes les images du candidat effectivement examinées et conserve les défauts observés.

Le protocole statistique de `evaluate.py` exige des annotations `human_identity_and_boxes`. Il mesure l'IoU, l'erreur de centre, les relations, la séparation et l'alpha. Les graines d'une même paire ne sont pas des observations indépendantes : le score regroupe les paires et calcule un intervalle de Wilson à 95 %.

Une revue par agent ne crée pas ces annotations humaines. Le petit benchmark courant ne satisfait pas à lui seul les 300 groupes indépendants exigés par défaut. La publication peut rester expérimentale, avec `spatial_quality_validated: false`.

In [ ]:
review = read_json(result_file("visual-review.json"), {})
required_ids = {r["id"] for r in predictions["rows"] if r["method"].startswith("candidate_")}
reviewed_ids = set(review.get("reviewed_ids", []))
table([{
    "méthode de revue": review.get("method", "absente"),
    "candidats présents": len(required_ids),
    "candidats couverts": len(required_ids & reviewed_ids),
    "conclusion déclarée": review.get("passed", "absente"),
}], ["méthode de revue", "candidats présents", "candidats couverts", "conclusion déclarée"])
if review:
    print(__import__("json").dumps(review, ensure_ascii=False, indent=2))

In [ ]:
RUN_PAIRED_GENERATION = False
RUN_HUMAN_EVALUATION = False
human_annotations = ARTIFACTS / "spatial/human-annotations.jsonl"

run_script("03-spatial-distillation/generate.py", [
    "--candidate", ARTIFACTS / "spatial/run/flow/best",
    "--dataset", ARTIFACTS / "spatial/dataset",
    "--cache", PROJECT / "cache/spatial",
    "--output", ARTIFACTS / "spatial/run/validation",
    "--include-teacher", "--limit", 24, "--steps", 30,
], enabled=RUN_PAIRED_GENERATION, gpu=True)
run_script("03-spatial-distillation/evaluate.py", [
    "--predictions", predictions_path, "--annotations", human_annotations,
    "--output", ARTIFACTS / "spatial/human-evaluation.json",
], enabled=RUN_HUMAN_EVALUATION)

## Positionner des PNG existants de manière déterministe

Pour placer exactement des assets existants dans des boîtes, `compose_layout.py` calcule directement la composition. Une génération libre reste probabiliste. Ces deux usages ont des critères de réussite différents ; la MSE d'embeddings ne fournit aucune garantie géométrique.

In [ ]:
RUN_COMPOSITION = False
run_script("03-spatial-distillation/compose_layout.py", [
    "--layout", ARTIFACTS / "spatial/example-layout.json",
    "--output", ARTIFACTS / "notebooks/composition.png",
], enabled=RUN_COMPOSITION)

**Suite :** [05 — LoRA Monster](05_lora_monster.ipynb).